In [1]:
import numpy as np
import pandas as pd

PIP = 0.0001
COST_PIPS = 1.4          # spread + slippage, round trip (from the specs)

def simulate_trade(bars, entry_i, side, stop, target, max_bars=None, exit_time=None):
    """
    bars:      DataFrame with open/high/low/close, index = bar start time (UTC)
    entry_i:   integer position of the bar we ENTER on (fill at its open)
    side:      +1 long, -1 short
    stop, target: absolute prices
    max_bars:  close at that bar's close if still open after this many bars
    exit_time: 'HH:MM' UTC, close at the open of the first bar at/after this time
    Returns a dict describing the trade, with P&L in pips after costs.
    """
    entry_px = bars["open"].iat[entry_i]
    n = len(bars)
    for k, i in enumerate(range(entry_i, n)):
        t = bars.index[i]
        o, h, l, c = bars["open"].iat[i], bars["high"].iat[i], bars["low"].iat[i], bars["close"].iat[i]

        # time exit: leave at this bar's open
        if exit_time is not None and k > 0 and t.strftime("%H:%M") >= exit_time:
            exit_px, reason = o, "time"
            break

        if side == 1:
            hit_stop, hit_tgt = l <= stop, h >= target
        else:
            hit_stop, hit_tgt = h >= stop, l <= target

        if hit_stop:                       # checked FIRST: worst case if both touched
            # if the bar opened beyond the stop (a gap), we fill at the open, not the stop
            gap = (o < stop) if side == 1 else (o > stop)
            exit_px, reason = (o if gap else stop), "stop"
            break
        if hit_tgt:
            exit_px, reason = target, "target"
            break
        if max_bars is not None and k + 1 >= max_bars:
            exit_px, reason = c, "max_bars"
            break
    else:
        exit_px, reason, i = bars["close"].iat[n - 1], "end_of_data", n - 1

    gross = side * (exit_px - entry_px) / PIP
    return {
        "entry_time": bars.index[entry_i], "exit_time": bars.index[i],
        "side": side, "entry": entry_px, "exit": exit_px, "reason": reason,
        "bars_held": i - entry_i + 1,
        "pips_gross": gross, "pips_net": gross - COST_PIPS,
        "risk_pips": abs(entry_px - stop) / PIP,
    }

In [2]:
def fake_bars(rows, start="2020-01-06 15:30", freq="15min"):
    idx = pd.date_range(start, periods=len(rows), freq=freq, tz="UTC")
    return pd.DataFrame(rows, columns=["open", "high", "low", "close"], index=idx)

def check(name, trade, reason, gross):
    ok = trade["reason"] == reason and abs(trade["pips_gross"] - gross) < 1e-6
    print(("PASS" if ok else "FAIL"), name, "->", trade["reason"], round(trade["pips_gross"], 1))
    return ok

results = []

# 1) long, target hit on bar 2  -> +20 pips
b = fake_bars([(1.1000, 1.1010, 1.0995, 1.1005), (1.1005, 1.1025, 1.1000, 1.1020)])
results.append(check("long target", simulate_trade(b, 0, 1, 1.0990, 1.1020), "target", 20))

# 2) long, stop hit  -> -10 pips
b = fake_bars([(1.1000, 1.1005, 1.0985, 1.0990)])
results.append(check("long stop", simulate_trade(b, 0, 1, 1.0990, 1.1020), "stop", -10))

# 3) SAME BAR touches both stop and target  -> must count as the stop (worst case)
b = fake_bars([(1.1000, 1.1030, 1.0980, 1.1000)])
results.append(check("same-bar both", simulate_trade(b, 0, 1, 1.0990, 1.1020), "stop", -10))

# 4) price GAPS through the stop  -> fill at the open (1.0980), worse than the stop: -20
b = fake_bars([(1.1000, 1.1005, 1.0995, 1.1000), (1.0980, 1.0985, 1.0975, 1.0982)])
results.append(check("gap through stop", simulate_trade(b, 0, 1, 1.0990, 1.1020), "stop", -20))

# 5) short, target hit  -> +20 pips
b = fake_bars([(1.1000, 1.1005, 1.0990, 1.0992), (1.0992, 1.0995, 1.0978, 1.0985)])
results.append(check("short target", simulate_trade(b, 0, -1, 1.1010, 1.0980), "target", 20))

# 6) nothing hit, max_bars=2  -> exit at bar 2's close (1.1004): +4
b = fake_bars([(1.1000, 1.1005, 1.0995, 1.1002), (1.1002, 1.1006, 1.0998, 1.1004), (1.1004, 1.1030, 1.1000, 1.1025)])
results.append(check("max bars", simulate_trade(b, 0, 1, 1.0990, 1.1020, max_bars=2), "max_bars", 4))

# 7) time exit at 16:00  -> leave at the 16:00 bar's open (1.1003): +3
b = fake_bars([(1.1000, 1.1005, 1.0995, 1.1002), (1.1002, 1.1006, 1.0998, 1.1003), (1.1003, 1.1030, 1.1000, 1.1025)])
results.append(check("time exit", simulate_trade(b, 0, 1, 1.0990, 1.1020, exit_time="16:00"), "time", 3))

print(f"\n{sum(results)}/{len(results)} passed")

PASS long target -> target 20.0
PASS long stop -> stop -10.0
PASS same-bar both -> stop -10.0
PASS gap through stop -> stop -20.0
PASS short target -> target 20.0
PASS max bars -> max_bars 4.0
PASS time exit -> time 3.0

7/7 passed
